# 06 Attention ? Transformer ??

## ????

- ?? Query?Key?Value ??????
- ?? scaled dot-product attention?
- ?????????
- ???? Transformer Block ?????????

## ????

??????????????????????????????????????

Attention ?????????????????????????????????????????? softmax ???????? Value ?????

## ????

- Query???????????
- Key???????? Query ??????
- Value??????????
- Attention weights????????????????
- Transformer Block???????????????????????

## ??????

?????????????????????????????????????

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# batch=1, seq_len=4, dim=3
x = torch.tensor([[[1.0, 0.0, 0.5],
                   [0.8, 0.2, 0.4],
                   [0.0, 1.0, 0.3],
                   [0.1, 0.9, 0.2]]])
print("input shape:", x.shape)

## ????

???????????????? `QK^T -> softmax -> ?? V` ???????scaled dot-product attention ????????? Query ? Key ??????????? Value?

### ?? Attention

????? `Q=K=V=x`??????????????????????????????????????

In [ ]:
def scaled_dot_product_attention(q, k, v):
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / np.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

attn_output, attn_weights = scaled_dot_product_attention(x, x, x)
print("attention output shape:", attn_output.shape)
print("attention weights:")
print(attn_weights[0])

In [ ]:
plt.figure(figsize=(4, 3.5))
plt.imshow(attn_weights[0].detach().numpy(), cmap="viridis")
plt.colorbar(label="weight")
plt.xlabel("key position")
plt.ylabel("query position")
plt.title("Attention weights")
plt.tight_layout()
plt.show()

### ? Transformer Block

???? `nn.MultiheadAttention` ??????? Transformer Block?????????? shape ????????? block ???????

In [ ]:
class TinyTransformerBlock(nn.Module):
    def __init__(self, embed_dim=8, num_heads=2, ff_dim=16):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_out, weights = self.attn(x, x, x, need_weights=True)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x, weights

block = TinyTransformerBlock(embed_dim=8, num_heads=2)
toy_tokens = torch.randn(2, 5, 8)
out, weights = block(toy_tokens)
print("input shape:", toy_tokens.shape)
print("output shape:", out.shape)
print("weights shape:", weights.shape)

## ????

??????????????????????????????????? query ??????? key ??????? Transformer Block ??? shape ?????????? block ???????

## ????

- softmax ???????????? key ??????
- `embed_dim` ??? `num_heads` ???
- ?? batch-first ? seq-first ??????
- ??????????????????

????????? attention scores?weights ?????? shape?

In [ ]:
row_sums = attn_weights[0].sum(dim=-1)
print("each attention row sums to:", row_sums)

## ????

**Q?Attention ???????????**  
A????????????????????????????????????????????????

**Q???? Transformer ????????**  
A??? block ???????????????????????????????

**Q??????? `sqrt(d_k)`?**  
A????????????????????? softmax ????

## ??

Attention ???????????????? Q/K/V???????????????? Transformer ??????